In [13]:
import pandas as pd

df = pd.read_parquet("data/intermediate/garantias_v5.parquet")
estados_unicos = sorted(df['programa'].dropna().astype(str).unique())

for estado in estados_unicos:
    print(estado)

ADQUISICION VEHICULOS
APOYO DESASTRES NATURALES
COBERTURAS DIFERENCIADAS
COMEX T-MEC
CREDIACTIVO
CREDICADENAS
CREDISUMINISTROS
CREDITO JOVEN
ECO CREDITO EMPRESARIAL
EFICIENCIA ENERGETICA
FINANCIAMIENTO A MICRONEGOCIOS
FINANCIAMIENTO DIGITAL
FORTALECIMIENTO IFNBS
GARANTIA AGIL
GARANTIA AUTOMATICA
GARANTIA SOBRE FIANZAS
GLOBAL PYME
IMPULSO NAFIN
MIPYME CREDITO COMERCIAL
MIPYME MUNICIPAL
MUJERES EMPRESARIAS
OTROS PROGRAMAS
PLAN MEXICO
PRODUCTO NAFIN EMPRESARIAL
PROVEEDORES SECTOR AUTOMOTRIZ
PROVEEDORES SECTOR ELECTRICO-ELECTRONICO
PYME HASTA 20 MDP
REACTIVACION ECONOMICA
RESTAURANTES
SECTOR MEDICO
SELECTIVA
SUSTITUCION PARQUE VEHICULAR
TRADICIONAL
TU PRIMER CREDITO
TURISMO


In [6]:
import pandas as pd
import numpy as np

# Cargar el archivo consolidado
ruta_parquet = "data/intermediate/garantias.parquet"
try:
    df = pd.read_parquet(ruta_parquet)
except FileNotFoundError:
    print(f"Error: No se encontró el archivo en {ruta_parquet}. Ejecuta primero el pipeline de consolidación.")
    exit()

# 1. Auditoría de Intermediarios
print("--- 1. AUDITORÍA DE INTERMEDIARIOS ---")
intermediarios_unicos = df['Nombre Intermediario'].unique()
print(f"Total de intermediarios únicos: {len(intermediarios_unicos)}")

# Buscar posibles espacios en blanco ocultos que fragmentarían los filtros
intermediarios_con_espacios = [i for i in intermediarios_unicos if type(i) == str and (i.startswith(' ') or i.endswith(' '))]
if intermediarios_con_espacios:
    print(f"⚠️ Alerta: Se detectaron intermediarios con espacios residuales: {intermediarios_con_espacios}")
else:
    print("✅ Intermediarios limpios sin espacios residuales.")

# 2. Auditoría de Programas (Identificar fugas del Regex)
print("\n--- 2. AUDITORÍA DE CLASIFICACIÓN DE PROGRAMAS ---")
programas_totales = df['Programa_Limpio'].value_counts()
print(f"Top 5 programas con más registros:\n{programas_totales.head(5)}")

# Aislar los registros que cayeron en la categoría por defecto
otros_programas = df[df['Programa_Limpio'] == 'OTROS PROGRAMAS']
if not otros_programas.empty:
    fugas_unicas = otros_programas['Programa'].unique()
    print(f"\n⚠️ Alerta: {len(fugas_unicas)} descripciones originales no fueron capturadas por el Regex y cayeron en 'OTROS PROGRAMAS'.")
    print("Ejemplos a revisar e incorporar al diccionario de expresiones regulares:")
    for fuga in fugas_unicas[:10]:
        print(f"  - {fuga}")
else:
    print("\n✅ El 100% de los programas fueron categorizados exitosamente.")

# 3. Auditoría de Estados (Evaluación de Estado_v2)
print("\n--- 3. AUDITORÍA GEOGRÁFICA (ESTADO_V2) ---")
cambios_estado = df[df['Estado'] != df['Estado_v2']]
print(f"Registros donde Estado_v2 corrigió o actualizó el Estado original: {len(cambios_estado)}")

if not cambios_estado.empty:
    print("Muestra de correcciones aplicadas:")
    muestra_cambios = cambios_estado[['Programa', 'Estado', 'Estado_v2']].drop_duplicates().head(5)
    print(muestra_cambios.to_string(index=False))

# Verificar nulos residuales
estados_nulos = df[df['Estado_v2'].isna() | (df['Estado_v2'] == '') | (df['Estado_v2'] == 'NAN')]
print(f"\nRegistros que quedaron sin estado asignado (Nulos en Estado_v2): {len(estados_nulos)}")

# 4. Validaciones de tipos de datos para rendimiento
print("\n--- 4. ROBUSTEZ DEL ESQUEMA ---")
print(df[['Nombre Intermediario', 'Programa_Limpio', 'Estado_v2']].dtypes)

--- 1. AUDITORÍA DE INTERMEDIARIOS ---
Total de intermediarios únicos: 28
✅ Intermediarios limpios sin espacios residuales.

--- 2. AUDITORÍA DE CLASIFICACIÓN DE PROGRAMAS ---
Top 5 programas con más registros:
Programa_Limpio
PRODUCTO NAFIN EMPRESARIAL    172774
IMPULSO NAFIN                  11850
FINANCIAMIENTO DIGITAL         11704
TURISMO                        10204
PLAN MEXICO                     7000
Name: count, dtype: int64

✅ El 100% de los programas fueron categorizados exitosamente.

--- 3. AUDITORÍA GEOGRÁFICA (ESTADO_V2) ---
Registros donde Estado_v2 corrigió o actualizó el Estado original: 1466
Muestra de correcciones aplicadas:
                                                               Programa           Estado        Estado_v2
                                BANORTE IMPULSO NAFIN EDO MEX 70% 2.17%           MEXICO ESTADO DE MEXICO
                                  MIFEL IMPULSO NAFIN EDO MEX 70% 2.17%           MEXICO ESTADO DE MEXICO
                             

In [10]:
import pandas as pd
import numpy as np

ruta_parquet = "data/intermediate/garantias.parquet"
df = pd.read_parquet(ruta_parquet)
df.columns = df.columns.str.strip()

print("--- 5. AUDITORÍA FINANCIERA Y TEMPORAL ---")
columnas_numericas = ['Monto Inicial', 'Monto Garantizado', 'Ultimo Saldo', 'Tasa']
columnas_fecha = ['Fecha Apertura', 'Fecha Registro']

print("Tipos de datos actuales:")
print(df[columnas_numericas + columnas_fecha].dtypes)

print("\n--- BÚSQUEDA DE ARTEFACTOS DE TEXTO ---")
for col in columnas_numericas:
    # Si la columna no es puramente numérica, buscamos qué caracteres la están ensuciando
    if not pd.api.types.is_numeric_dtype(df[col]):
        # Extraer una muestra de los valores que contienen letras, espacios o símbolos extraños
        valores_texto = df[df[col].astype(str).str.contains(r'[^\d\.-]', na=False, regex=True)][col].unique()
        print(f"⚠️ {col} contiene caracteres no numéricos. Muestra: {valores_texto[:5]}")
    else:
        print(f"✅ {col} es un tipo numérico puro.")

print("\n--- INTEGRIDAD TEMPORAL ---")
for col in columnas_fecha:
    if not pd.api.types.is_datetime64_any_dtype(df[col]):
        print(f"⚠️ {col} no es tipo datetime. Formato actual: {df[col].dtype}")
    else:
        rango = df[col].agg(['min', 'max'])
        print(f"✅ {col} confirmada. Rango: {rango['min'].date()} a {rango['max'].date()}")

--- 5. AUDITORÍA FINANCIERA Y TEMPORAL ---
Tipos de datos actuales:
Monto Inicial                 int64
Monto Garantizado             int64
Ultimo Saldo                float64
Tasa                       category
Fecha Apertura       datetime64[us]
Fecha Registro       datetime64[us]
dtype: object

--- BÚSQUEDA DE ARTEFACTOS DE TEXTO ---
✅ Monto Inicial es un tipo numérico puro.
✅ Monto Garantizado es un tipo numérico puro.
✅ Ultimo Saldo es un tipo numérico puro.
⚠️ Tasa contiene caracteres no numéricos. Muestra: ['TIIE 28', 'TASA FIJA', 'SIN TASA CON COMISION', 'TASA FIJA Y/O VARIABLE', 'TASA SOFR']
Categories (9, str): ['SIN TASA CON COMISION', 'TARIFA', 'TASA BASE', 'TASA FIJA', ..., 'TASA SOFR', 'TIIE 28', 'TIIE DE FONDEO A UN DIA', 'TIIE DE FONDEO COMPUESTA POR ADELANTADO']

--- INTEGRIDAD TEMPORAL ---
✅ Fecha Apertura confirmada. Rango: 2024-03-22 a 2026-08-28
✅ Fecha Registro confirmada. Rango: 2025-01-03 a 2026-08-31


In [11]:
import pandas as pd

df = pd.read_parquet("data/intermediate/garantias.parquet")

print("PEGA ESTO EN EL PROMPT DE CLAUDE:")
print(f"- Total Registros: {len(df)}")
print(f"- Rango Temporal: {df['Fecha Apertura'].min().date()} a {df['Fecha Apertura'].max().date()}")
print(f"- Bancos de 2do Piso: {df['Banco(Nafin/BCMXT)'].nunique()} ({', '.join(df['Banco(Nafin/BCMXT)'].dropna().unique())})")
print(f"- Intermediarios Únicos: {df['Nombre Intermediario'].nunique()}")
print(f"- Programas Limpios Únicos: {df['Programa_Limpio'].nunique()}")
print(f"- Estados Únicos: {df['Estado_v2'].nunique()}")
print(f"- Monto Inicial Total: ${df['Monto Inicial'].sum():,.2f}")
print(f"- Saldo Actual Total: ${df['Ultimo Saldo'].sum():,.2f}")

PEGA ESTO EN EL PROMPT DE CLAUDE:
- Total Registros: 236565
- Rango Temporal: 2024-03-22 a 2026-08-28
- Bancos de 2do Piso: 2 (NAFIN, BANCOMEXT)
- Intermediarios Únicos: 28
- Programas Limpios Únicos: 35
- Estados Únicos: 34
- Monto Inicial Total: $666,833,972,197.00
- Saldo Actual Total: $430,022,268,148.00


In [1]:
import pandas as pd

pd.read_parquet("data/intermediate/garantias.parquet")['Fecha Registro'].max()

Timestamp('2026-08-31 00:00:00')

In [1]:
import pandas as pd

pd.options.display.float_format = '{:.3f}'.format

test =pd.read_parquet("data/raw/garantias26_0926.parquet")

In [3]:
test

,BANCO,ESTADO,ESTRATO_DESCRIPCION,INT_RAZON_SOCIAL,Fecha Consulta (Saldos) (MA),Fecha Registro (MDA),Moneda Id (Saldo),NOMBRE_EMPRESA,PROGRAMA_DESCRIPCION,RFC_EMPRESA,Valor Tasa Interes,CONSOLID CONTING (AGR),CONSOLID INDUC (AGR),CONTEO_GARS (SUMA),Monto Credito Mn (SUMA),Monto Garantizado (Saldo) (SUMA)
0,NAFIN,PUEBLA,MICRO EMPRESA,FIDEICOMISO PARA EL AHORRO DE ENERGIA ELECTRICA,202606,20240626,1,GRUPO AGSO SA DE CV,FIDE AHORRO ENERGETICO EMPRESARIAL MASIVO FASE...,GAG-220523-536,0.050,40677.994,40677.994,1.000,227800.800,35309.120
1,BANCOMEXT,JALISCO,EMPRESA GRANDE,"BANCOMEXT/BANCO SABADELL, S.A., I.B.M.",202601,20250404,1,MB MOTOS S A DE C V,BANCOMEXT/SABADELL COMEX TMEC 2024 60% 2.50%,MMO-020404-R92,5.000,4200000.000,7000000.000,1.000,7000000.000,4200000.000
2,NAFIN,JALISCO,PEQUEÑA EMPRESA,BBV BANCOMER,202603,20240828,1,RUTH PATRICIA JANAKO GARCIA AZANO,BBVA PRODUCTO NAFIN EMPRESARIAL FII 2024,GAAR-620208-RT3,15.250,4036875.020,8073750.040,1.000,10765000.000,5382500.000
3,NAFIN,MEXICO,MICRO EMPRESA,HSBC MEXICO,202602,20250829,1,KARINA MARIBEL MACIAS RAMIREZ,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2025,MARK-950223-G13,30.000,213296.780,426579.065,1.000,440000.000,220000.000
4,NAFIN,JALISCO,PEQUEÑA EMPRESA,BANCO MERCANTIL DEL NORTE,202601,20250331,1,ACP BLINDAJE SA DE CV,BANORTE PRODUCTO NAFIN EMPRESARIAL 2025,ABL-120312-8HA,5.000,1600173.188,3200173.188,1.000,6000000.000,3000000.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1266408,NAFIN,DURANGO,PEQUEÑA EMPRESA,HSBC MEXICO,202608,20260831,1,SILVIA LAURA CUEVAS MORILLON,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2026,CUMS-600405-I84,23.000,0.500,1.000,1.000,300000.000,0.500
1266409,NAFIN,MEXICO,PEQUEÑA EMPRESA,HSBC MEXICO,202608,20260831,1,INDUSTRIEL IPM SA DE CV,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2026,IIP-191010-T8A,25.000,500037.223,1000037.223,1.000,1000000.000,500000.000
1266410,NAFIN,PUEBLA,PEQUEÑA EMPRESA,HSBC MEXICO,202608,20260831,1,CASTILLO OJEDA ANTONIA,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2026,CAOA-510613-T6A,25.000,500037.223,1000037.223,1.000,2000000.000,500000.000
1266411,NAFIN,PUEBLA,MICRO EMPRESA,HSBC MEXICO,202608,20260831,1,DEBORAH HERRERA HERRERA,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2026,HEHD-600628-I18,15.250,110008.189,220008.189,1.000,220000.000,110000.000


In [27]:
test

,BANCO,ESTADO,INT_RAZON_SOCIAL,Fecha Consulta (Saldos) (MA),Fecha Registro (MDA),Moneda Id (Saldo),NOMBRE_EMPRESA,Porcentaje Comision Garantia,PROGRAMA_DESCRIPCION,RFC_EMPRESA,Valor Tasa Interes,CONSOLID CONTING (AGR),CONSOLID INDUC (AGR),CONTEO_GARS (SUMA),Monto Credito Mn (SUMA),Monto Garantizado (Saldo) (SUMA)
0,BANCOMEXT,CIUDAD DE MEXICO,BANCOMEXTHSBC,202402,20230821,1,BANCOMEXTHSBC,4.200,BANCOMEXT/HSBC TURISMO 60% 4.20%,BIN-950125-4W1,16.000,16396.986,27328.310,1.000,30000.000,18000.000
1,BANCOMEXT,GUANAJUATO,BANCOMEXT/BAJIO,202604,20251002,1,BANCOMEXT/BAJIO,2.600,BANCOMEXT/BAJIO PROV DEL SECTOR AUTOMOTRIZ 202...,NO CONTIENE RFC,6.660,2400000.000,3000000.000,1.000,3000000.000,800000.000
2,NAFIN,NUEVO LEON,BANCO REGIONAL DE MONTERREY,202401,20230622,1,BANCO REGIONAL DE MONTERREY,8.790,BANREGIO PRODUCTO NAFIN EMPRESARIAL FII 2023,BRM-940216-NRA,21.000,42948.665,85897.330,1.000,100000.000,103.170
3,NAFIN,NaN,BANAMEX,202512,20240829,1,BANAMEX,4.200,CITIBANAMEX PRODUCTO NAFIN EMPRESARIAL FII 2024,BNM-840515-VB1,8.800,1071699.346,2143368.801,1.000,5200000.000,0.500
4,NAFIN,NUEVO LEON,BANCO REGIONAL DE MONTERREY,202406,20240110,1,BANCO REGIONAL DE MONTERREY,4.120,BANREGIO PRODUCTO NAFIN EMPRESARIAL 2024,BRM-940216-NRA,7.000,431731.630,863463.260,1.000,1500000.000,368968.250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1803781,NAFIN,CIUDAD DE MEXICO,HSBC MEXICO,202608,20260831,1,HSBC MEXICO,2.170,HSBC IMPULSO NAFIN- AGUASCALIENTES 70% 2.17%,HMI-950125-KG8,14.750,511038.042,730038.042,1.000,730000.000,511000.000
1803782,NAFIN,CIUDAD DE MEXICO,HSBC MEXICO,202608,20260831,1,HSBC MEXICO,2.170,HSBC IMPULSO NAFIN- JALISCO 70% 2.17%,HMI-950125-KG8,14.750,1127083.902,1610083.902,3.000,1610000.000,1127000.000
1803783,NAFIN,CIUDAD DE MEXICO,HSBC MEXICO,202608,20260831,1,HSBC MEXICO,5.850,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2026,HMI-950125-KG8,13.000,0.500,1.000,1.000,2600000.000,0.500
1803784,NAFIN,CIUDAD DE MEXICO,HSBC MEXICO,202608,20260831,1,HSBC MEXICO,5.010,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2026,HMI-950125-KG8,15.250,345025.684,690025.684,2.000,690000.000,345000.000


In [ ]:
pd.read_parquet("data/raw/garantias_dw_0926.parquet")['RFC_EMPRESA']

Index(['BANCO', 'ESTADO', 'INT_RAZON_SOCIAL', 'Fecha Consulta (Saldos) (MA)',
       'Fecha Registro (MDA)', 'Moneda Id (Saldo)', 'NOMBRE_EMPRESA',
       'Porcentaje Comision Garantia', 'PROGRAMA_DESCRIPCION', 'RFC_EMPRESA',
       'Valor Tasa Interes', 'CONSOLID CONTING (AGR)', 'CONSOLID INDUC (AGR)',
       'CONTEO_GARS (SUMA)', 'Monto Credito Mn (SUMA)',
       'Monto Garantizado (Saldo) (SUMA)'],
      dtype='str')

In [5]:
test.to_csv("data/raw/garantias26_0926.csv", index=False)

In [17]:
import pandas as pd

pd.options.display.float_format = '{:.3f}'.format

pd.read_parquet("data/raw/garantias26_0926.parquet")

,BANCO,ESTADO,ESTRATO_DESCRIPCION,INT_RAZON_SOCIAL,Fecha Consulta (Saldos) (MA),Fecha Registro (MDA),Moneda Id (Saldo),NOMBRE_EMPRESA,PROGRAMA_DESCRIPCION,RFC_EMPRESA,Valor Tasa Interes,CONSOLID CONTING (AGR),CONSOLID INDUC (AGR),CONTEO_GARS (SUMA),Monto Credito Mn (SUMA),Monto Garantizado (Saldo) (SUMA)
0,NAFIN,PUEBLA,MICRO EMPRESA,FIDEICOMISO PARA EL AHORRO DE ENERGIA ELECTRICA,202606,20240626,1,GRUPO AGSO SA DE CV,FIDE AHORRO ENERGETICO EMPRESARIAL MASIVO FASE...,GAG-220523-536,0.050,40677.994,40677.994,1.000,227800.800,35309.120
1,BANCOMEXT,JALISCO,EMPRESA GRANDE,"BANCOMEXT/BANCO SABADELL, S.A., I.B.M.",202601,20250404,1,MB MOTOS S A DE C V,BANCOMEXT/SABADELL COMEX TMEC 2024 60% 2.50%,MMO-020404-R92,5.000,4200000.000,7000000.000,1.000,7000000.000,4200000.000
2,NAFIN,JALISCO,PEQUEÑA EMPRESA,BBV BANCOMER,202603,20240828,1,RUTH PATRICIA JANAKO GARCIA AZANO,BBVA PRODUCTO NAFIN EMPRESARIAL FII 2024,GAAR-620208-RT3,15.250,4036875.020,8073750.040,1.000,10765000.000,5382500.000
3,NAFIN,MEXICO,MICRO EMPRESA,HSBC MEXICO,202602,20250829,1,KARINA MARIBEL MACIAS RAMIREZ,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2025,MARK-950223-G13,30.000,213296.780,426579.065,1.000,440000.000,220000.000
4,NAFIN,JALISCO,PEQUEÑA EMPRESA,BANCO MERCANTIL DEL NORTE,202601,20250331,1,ACP BLINDAJE SA DE CV,BANORTE PRODUCTO NAFIN EMPRESARIAL 2025,ABL-120312-8HA,5.000,1600173.188,3200173.188,1.000,6000000.000,3000000.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1266408,NAFIN,DURANGO,PEQUEÑA EMPRESA,HSBC MEXICO,202608,20260831,1,SILVIA LAURA CUEVAS MORILLON,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2026,CUMS-600405-I84,23.000,0.500,1.000,1.000,300000.000,0.500
1266409,NAFIN,MEXICO,PEQUEÑA EMPRESA,HSBC MEXICO,202608,20260831,1,INDUSTRIEL IPM SA DE CV,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2026,IIP-191010-T8A,25.000,500037.223,1000037.223,1.000,1000000.000,500000.000
1266410,NAFIN,PUEBLA,PEQUEÑA EMPRESA,HSBC MEXICO,202608,20260831,1,CASTILLO OJEDA ANTONIA,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2026,CAOA-510613-T6A,25.000,500037.223,1000037.223,1.000,2000000.000,500000.000
1266411,NAFIN,PUEBLA,MICRO EMPRESA,HSBC MEXICO,202608,20260831,1,DEBORAH HERRERA HERRERA,HSBC PRODUCTO NAFIN EMPRESARIAL RBB 2026,HEHD-600628-I18,15.250,110008.189,220008.189,1.000,220000.000,110000.000


In [25]:
# Cargar el DataFrame
df = pd.read_parquet("data/raw/garantias26_0926.parquet")

# 1. Contar duplicados en 'RFC_EMPRESA'
# Total de filas con RFC repetido (manteniendo todas las ocurrencias repetidas)
cols_dup = ['INT_RAZON_SOCIAL', 'Fecha Registro (MDA)', 'PROGRAMA_DESCRIPCION', 'RFC_EMPRESA', 'Valor Tasa Interes']

total_duplicados = df.duplicated(subset=cols_dup, keep=False).sum()
print(f"Total de registros duplicados: {total_duplicados}")

conteo_duplicados = df.value_counts(subset=cols_dup)
grupos_duplicados = conteo_duplicados[conteo_duplicados > 1].reset_index(name='conteo')
grupos_duplicados

Total de registros duplicados: 1245774


,INT_RAZON_SOCIAL,Fecha Registro (MDA),PROGRAMA_DESCRIPCION,RFC_EMPRESA,Valor Tasa Interes,conteo
0,ACE FIANZAS MONTERREY S.A.,20250228,CHUBB FIANZAS MONTERREY GTIA SOBRE FIANZAS PAR...,HSM-970707-RA5,0.000,12
1,ACE FIANZAS MONTERREY S.A.,20250228,CHUBB FIANZAS MONTERREY GTIA SOBRE FIANZAS PAR...,NDM-000515-T51,0.000,12
2,FIDEICOMISO PARA EL AHORRO DE ENERGIA ELECTRICA,20240626,FIDE AHORRO ENERGETICO EMPRESARIAL MASIVO FASE...,GAG-220523-536,0.050,8
3,"BANCOMEXT/BANCO SABADELL, S.A., I.B.M.",20250404,BANCOMEXT/SABADELL COMEX TMEC 2024 60% 2.50%,MMO-020404-R92,5.000,8
4,BBV BANCOMER,20240828,BBVA PRODUCTO NAFIN EMPRESARIAL FII 2024,GAAR-620208-RT3,15.250,8
...,...,...,...,...,...,...
194348,BBV BANCOMER,20260731,BBVA PROYECTO SECTORIAL PLAN MEXICO 70% 2.30%,VPR-230705-AF0,14.500,2
194349,BBV BANCOMER,20260731,BBVA PROYECTO SECTORIAL PLAN MEXICO 70% 2.30%,PEGS-621108-2W7,14.500,2
194350,BBV BANCOMER,20260731,BBVA PROYECTO SECTORIAL PLAN MEXICO 70% 2.30%,TSP-170427-KD6,14.500,2
194351,BBV BANCOMER,20260731,BBVA PROYECTO SECTORIAL PLAN MEXICO 70% 2.30%,PAB-101230-NZ2,14.500,2


In [26]:
# 2. Consultar todos los registros de un RFC específico
def buscar_registros(df, razon_social=None, fecha_reg=None, programa=None, rfc=None, tasa=None):
    cond = pd.Series(True, index=df.index)
    if razon_social is not None:
        cond &= df['INT_RAZON_SOCIAL'].astype(str).str.strip() == str(razon_social).strip()
    if fecha_reg is not None:
        cond &= df['Fecha Registro (MDA)'].astype(str).str.strip() == str(fecha_reg).strip()
    if programa is not None:
        cond &= df['PROGRAMA_DESCRIPCION'].astype(str).str.strip() == str(programa).strip()
    if rfc is not None:
        cond &= df['RFC_EMPRESA'].astype(str).str.strip() == str(rfc).strip()
    if tasa is not None:
        cond &= df['Valor Tasa Interes'].astype(str).str.strip() == str(tasa).strip()
    return df[cond]
    

registros = buscar_registros(
    df,
    razon_social="BBV BANCOMER",
    fecha_reg=20250730,
    programa="BBVA PRODUCTO NAFIN EMPRESARIAL 2025",
    rfc="EOG-150704-RM1",
    tasa=14.250
)
print(f"\nRegistros encontrados: {len(registros)}")
registros


Registros encontrados: 8


,BANCO,ESTADO,ESTRATO_DESCRIPCION,INT_RAZON_SOCIAL,Fecha Consulta (Saldos) (MA),Fecha Registro (MDA),Moneda Id (Saldo),NOMBRE_EMPRESA,PROGRAMA_DESCRIPCION,RFC_EMPRESA,Valor Tasa Interes,CONSOLID CONTING (AGR),CONSOLID INDUC (AGR),CONTEO_GARS (SUMA),Monto Credito Mn (SUMA),Monto Garantizado (Saldo) (SUMA)
109690,NAFIN,TABASCO,PEQUEÑA EMPRESA,BBV BANCOMER,202606,20250730,1,ESP OIL AND GAS ENER GY SA DE CV,BBVA PRODUCTO NAFIN EMPRESARIAL 2025,EOG-150704-RM1,14.250,809945.780,1619891.560,1.000,1822350.000,911175.000
172096,NAFIN,TABASCO,PEQUEÑA EMPRESA,BBV BANCOMER,202601,20250730,1,ESP OIL AND GAS ENER GY SA DE CV,BBVA PRODUCTO NAFIN EMPRESARIAL 2025,EOG-150704-RM1,14.250,867749.900,1735499.800,1.000,1822350.000,911175.000
280556,NAFIN,TABASCO,PEQUEÑA EMPRESA,BBV BANCOMER,202602,20250730,1,ESP OIL AND GAS ENER GY SA DE CV,BBVA PRODUCTO NAFIN EMPRESARIAL 2025,EOG-150704-RM1,14.250,856733.620,1713467.240,1.000,1822350.000,911175.000
584522,NAFIN,TABASCO,PEQUEÑA EMPRESA,BBV BANCOMER,202603,20250730,1,ESP OIL AND GAS ENER GY SA DE CV,BBVA PRODUCTO NAFIN EMPRESARIAL 2025,EOG-150704-RM1,14.250,845582.160,1691164.320,1.000,1822350.000,911175.000
821997,NAFIN,TABASCO,PEQUEÑA EMPRESA,BBV BANCOMER,202605,20250730,1,ESP OIL AND GAS ENER GY SA DE CV,BBVA PRODUCTO NAFIN EMPRESARIAL 2025,EOG-150704-RM1,14.250,821850.600,1643701.200,1.000,1822350.000,911175.000
889053,NAFIN,TABASCO,PEQUEÑA EMPRESA,BBV BANCOMER,202604,20250730,1,ESP OIL AND GAS ENER GY SA DE CV,BBVA PRODUCTO NAFIN EMPRESARIAL 2025,EOG-150704-RM1,14.250,833289.735,1666579.470,1.000,1822350.000,911175.000
1138259,NAFIN,TABASCO,PEQUEÑA EMPRESA,BBV BANCOMER,202607,20250730,1,ESP OIL AND GAS ENER GY SA DE CV,BBVA PRODUCTO NAFIN EMPRESARIAL 2025,EOG-150704-RM1,14.250,798220.195,1596440.390,1.000,1822350.000,911175.000
1226176,NAFIN,TABASCO,PEQUEÑA EMPRESA,BBV BANCOMER,202608,20250730,1,ESP OIL AND GAS ENER GY SA DE CV,BBVA PRODUCTO NAFIN EMPRESARIAL 2025,EOG-150704-RM1,14.250,786034.765,1572069.530,1.000,1822350.000,911175.000


In [ ]:
df=pd.read_parquet("data/raw/garantias26_0926.parquet")


In [19]:
df.to_csv("data/raw/garantias26_0926.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_parquet("data/raw/garantias26_0926.parquet")

resultado = (
    df[df["BANCO"] == "NAFIN"]
    .groupby(["BANCO", "Fecha Consulta (Saldos) (MA)"], as_index=False)
    .agg(CONSOLIDADO_MILES=("CONSOLID CONTING (AGR)", lambda x: x.sum() / 1_000_000))
)

In [2]:
resultado

,BANCO,Fecha Consulta (Saldos) (MA),CONSOLIDADO_MILES
0,NAFIN,202601,133264.356441
1,NAFIN,202602,134130.538393
2,NAFIN,202603,134881.757003
3,NAFIN,202604,136463.733595
4,NAFIN,202605,138277.415675
5,NAFIN,202606,141484.878064
6,NAFIN,202607,142387.575870
7,NAFIN,202608,142240.405746


In [3]:
import pandas as pd
from pathlib import Path

archivos = [
    "data/raw/garantias24_0926.parquet",
    "data/raw/garantias25_0926.parquet",
    "data/raw/garantias26_0926.parquet"
]

lista_dfs = [pd.read_parquet(archivo) for archivo in archivos]

df_consolidado = pd.concat(lista_dfs, ignore_index=True)

ruta_salida = Path("data/intermediate")
ruta_salida.mkdir(parents=True, exist_ok=True)

df_consolidado.to_parquet("data/intermediate/garantias_0926.parquet", index=False)

In [4]:
pd.read_parquet("data/raw/garantias26_0926.parquet").info()

<class 'pandas.DataFrame'>
RangeIndex: 1266413 entries, 0 to 1266412
Data columns (total 16 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   BANCO                             1266413 non-null  str    
 1   ESTADO                            1266413 non-null  str    
 2   ESTRATO_DESCRIPCION               1266413 non-null  str    
 3   INT_RAZON_SOCIAL                  1266413 non-null  str    
 4   Fecha Consulta (Saldos) (MA)      1266413 non-null  int64  
 5   Fecha Registro (MDA)              1266413 non-null  int64  
 6   Moneda Id (Saldo)                 1266413 non-null  int64  
 7   NOMBRE_EMPRESA                    1266413 non-null  str    
 8   PROGRAMA_DESCRIPCION              1266413 non-null  str    
 9   RFC_EMPRESA                       1266413 non-null  str    
 10  Valor Tasa Interes                1266413 non-null  float64
 11  CONSOLID CONTING (AGR)            1266413 non-nu

In [7]:
pd.read_parquet("data/raw/garantias26_0926.parquet")['BANCO'].unique()

<ArrowStringArray>
['NAFIN', 'BANCOMEXT']
Length: 2, dtype: str

In [8]:
import pandas as pd

df = pd.read_parquet("data/raw/garantias26_0926.parquet")

cols_no_float = df.select_dtypes(exclude=['float64']).columns

valores_unicos_dict = {}

for col in cols_no_float:
    unicos = df[col].unique()
    valores_unicos_dict[col] = unicos
    
    print(f"Columna: {col} | Valores únicos: {len(unicos)}")
    print(unicos[:15])
    print("-" * 50)

Columna: BANCO | Valores únicos: 2
<ArrowStringArray>
['NAFIN', 'BANCOMEXT']
Length: 2, dtype: str
--------------------------------------------------
Columna: ESTADO | Valores únicos: 32
<ArrowStringArray>
[          'PUEBLA',          'JALISCO',           'MEXICO',
          'DURANGO',       'NUEVO LEON',        'QUERETARO',
           'SONORA',  'SAN LUIS POTOSI',          'CHIAPAS',
 'CIUDAD DE MEXICO',       'TAMAULIPAS',     'QUINTANA ROO',
        'MICHOACAN',        'CHIHUAHUA',         'CAMPECHE']
Length: 15, dtype: str
--------------------------------------------------
Columna: ESTRATO_DESCRIPCION | Valores únicos: 4
<ArrowStringArray>
['MICRO EMPRESA', 'EMPRESA GRANDE', 'PEQUEÑA EMPRESA', 'MEDIANA EMPRESA']
Length: 4, dtype: str
--------------------------------------------------
Columna: INT_RAZON_SOCIAL | Valores únicos: 49
<ArrowStringArray>
['FIDEICOMISO PARA EL AHORRO DE ENERGIA ELECTRICA',
          'BANCOMEXT/BANCO SABADELL, S.A., I.B.M.',
                              

In [1]:
import pandas as pd

# Simulamos la constante global de tu entorno
ORDEN_ESTRATO_PREFERIDO = ["MICRO", "PEQUENA", "MEDIANA", "GRANDE"]

def _norm_estrato(v: str) -> str:
    v = str(v).upper().strip()
    for a, b in (("Á", "A"), ("É", "E"), ("Í", "I"), ("Ó", "O"), ("Ú", "U"), ("Ñ", "N")):
        v = v.replace(a, b)
    return v

# Simulamos cómo llega el texto desde la base de datos a veces (espacios, plurales ocultos)
data = {
    "estrato": [" Mediana ", "Pequeñas", "  MICRO", "GRANDE", "Otro"],
    "saldo": [1000, 500, 2000, 300, 100]
}

df_test = pd.DataFrame(data)

print("=== DATOS ORIGINALES ===")
print(df_test)
print("\n")

# LÓGICA 1: La que usamos con el Diccionario Exacto
pesos_exactos = {"MICRO": 1, "PEQUENA": 2, "MEDIANA": 3, "GRANDE": 4}
df_test_dict = df_test.copy()
df_test_dict["peso_orden"] = df_test_dict["estrato"].apply(lambda x: pesos_exactos.get(_norm_estrato(x), 99))
df_test_dict = df_test_dict.sort_values("peso_orden")

print("=== RESULTADO CON DICCIONARIO (Coincidencia Exacta) ===")
print(df_test_dict[["estrato", "peso_orden"]])
print("Nota: Observa cómo 'Pequeñas' y 'Mediana' con espacio toman valor 99 porque no hacen match exacto.\n")

# LÓGICA 2: La que sugerí con la Subcadena
def _peso_subcadena(val):
    s = str(val).upper()
    if "MICRO" in s: return 1
    if "PEQ" in s: return 2
    if "MEDIAN" in s: return 3
    if "GRAND" in s: return 4
    return 99

df_test_sub = df_test.copy()
df_test_sub["peso_orden"] = df_test_sub["estrato"].apply(_peso_subcadena)
df_test_sub = df_test_sub.sort_values("peso_orden")

print("=== RESULTADO CON FUNCIÓN (Subcadena infalible) ===")
print(df_test_sub[["estrato", "peso_orden"]])
print("Nota: Aquí el orden matemático es perfecto: 1, 2, 3, 4, 99.")

=== DATOS ORIGINALES ===
     estrato  saldo
0   Mediana    1000
1   Pequeñas    500
2      MICRO   2000
3     GRANDE    300
4       Otro    100


=== RESULTADO CON DICCIONARIO (Coincidencia Exacta) ===
     estrato  peso_orden
2      MICRO           1
0   Mediana            3
3     GRANDE           4
1   Pequeñas          99
4       Otro          99
Nota: Observa cómo 'Pequeñas' y 'Mediana' con espacio toman valor 99 porque no hacen match exacto.

=== RESULTADO CON FUNCIÓN (Subcadena infalible) ===
     estrato  peso_orden
2      MICRO           1
1   Pequeñas           2
0   Mediana            3
3     GRANDE           4
4       Otro          99
Nota: Aquí el orden matemático es perfecto: 1, 2, 3, 4, 99.


In [ ]:
import pyodbc
import pandas as pd
import os
import warnings

# Ignorar las advertencias de pandas sobre el uso directo de conexiones pyodbc
warnings.filterwarnings('ignore', category=UserWarning)

def mdb_a_parquet(ruta_mdb, directorio_salida):
    # Asegurar que el directorio de salida exista
    os.makedirs(directorio_salida, exist_ok=True)
    
    # Obtener la ruta absoluta del archivo
    ruta_absoluta_mdb = os.path.abspath(ruta_mdb)
    
    # Cadena de conexión para Microsoft Access
    conn_str = (
        r"Driver={Microsoft Access Driver (*.mdb, *.accdb)};"
        f"DBQ={ruta_absoluta_mdb};"
    )
    
    conn = None
    try:
        print(f"Conectando a {ruta_absoluta_mdb}...")
        conn = pyodbc.connect(conn_str)
        cursor = conn.cursor()
        
        # Extraer los nombres de todas las tablas de usuarios (excluyendo las del sistema)
        tablas = [table.table_name for table in cursor.tables(tableType='TABLE')]
        print(f"Se encontraron {len(tablas)} tablas: {tablas}\n")
        
        for tabla in tablas:
            print(f"Procesando tabla: [{tabla}]...")
            
            # Leer la tabla completa hacia un DataFrame
            query = f"SELECT * FROM [{tabla}]"
            df = pd.read_sql(query, conn)
            
            # Definir la ruta del archivo parquet resultante
            ruta_parquet = os.path.join(directorio_salida, f"{tabla}.parquet")
            
            # Convertir a Parquet
            df.to_parquet(ruta_parquet, engine='pyarrow', index=False)
            print(f" ✓ Guardado exitosamente: {ruta_parquet}")
            
    except pyodbc.Error as e:
        print(f"\nError de ODBC (Asegúrate de tener el driver de Access instalado): {e}")
    except Exception as e:
        print(f"\nOcurrió un error inesperado: {e}")
    finally:
        # Cerrar siempre la conexión
        if conn is not None:
            conn.close()
            print("\nConexión cerrada.")

if __name__ == "__main__":
    # Directorio donde se encuentra este script (carpeta 'scripts')
    script_dir = os.path.dirname(os.path.abspath(__file__))
    
    # Directorio padre ('Garantias') y ruta hacia 'data/raw'
    base_dir = os.path.dirname(script_dir)
    carpeta_destino = os.path.join(base_dir, "data", "raw")
    archivo_origen = os.path.join(carpeta_destino, "DMF_RFC_2018_2026.mdb")
    
    mdb_a_parquet(archivo_origen, carpeta_destino)

Conectando a c:\Users\e_edobru\Desktop\Bancomext\Garantias\scripts\data\raw\DMF_RFC_2018_2026.mdb...

Error de ODBC (Asegúrate de tener el driver de Access instalado): ('HY000', "[HY000] [Microsoft][Controlador ODBC Microsoft Access] No se pudo encontrar el archivo '(desconocido)'. (-1811) (SQLDriverConnect); [HY000] [Microsoft][Controlador ODBC Microsoft Access] No se pudo encontrar el archivo '(desconocido)'. (-1811)")


In [1]:
import pandas as pd

df = pd.read_parquet("data/intermediate/garantias_v5_part/anio_mes=2026-06/176cf3ebae6240feacda790f36379964-0.parquet")

df['Fecha Registro (MDA)'] = df['Fecha Registro (MDA)'].astype(str)


df[df['Fecha Registro (MDA)'].str.startswith('2026')]['rfc'].nunique()

34832

In [9]:
import pandas as pd

# Leer el archivo
df = pd.read_parquet("data/raw/rfc_sin_conteo.parquet")

# Agrupar por la columna de fecha y contar todos los RFC (conteo total)
conteo_rfc_por_mes = (
    df.groupby("Fecha o Rango de Consulta (MA)")["RFC Empresa / Acreditado"]
    .count()
    .reset_index()
)

# Renombrar la columna del conteo para mayor claridad
conteo_rfc_por_mes = conteo_rfc_por_mes.rename(
    columns={"RFC Empresa / Acreditado": "Cantidad Total de RFC"}
)

# Mostrar el resultado
print(conteo_rfc_por_mes)

     Fecha o Rango de Consulta (MA)  Cantidad Total de RFC
0                            201801                    804
1                            201802                  26066
2                            201803                  39668
3                            201804                  40151
4                            201805                  40927
..                              ...                    ...
99                           202604                    939
100                          202605                   1056
101                          202606                   1497
102                          202607                   1516
103                          202608                   1662

[104 rows x 2 columns]


In [28]:
pd.read_parquet("data/intermediate/garantias_v5_part/anio_mes=2026-08/176cf3ebae6240feacda790f36379964-0.parquet")["rfc"].nunique()

114491

In [2]:
import pandas as pd

pd.read_parquet("data/intermediate/garantias_v5_part/anio_mes=2026-06/176cf3ebae6240feacda790f36379964-0.parquet")

,banco,estrato,Fecha Registro (MDA),NOMBRE_EMPRESA,rfc,tasa,saldo,CONSOLID INDUC (AGR),CONTEO_GARS (SUMA),monto_colocado,periodo,programa,estado,intermediario
0,NAFIN,MICRO EMPRESA,20240626,GRUPO AGSO SA DE CV,GAG-220523-536,0.05,4.067799e+04,4.067799e+04,1.0,227800.80,2026-06-01,EFICIENCIA ENERGETICA Y RENOVABLES,PUEBLA,FIDE
1,BANCOMEXT,PEQUEÑA EMPRESA,20250714,OPERADORA LA PALAPITA S DE RL DE CV,OPA-230110-990,16.50,7.082092e+05,1.180349e+06,1.0,1500000.00,2026-06-01,COMERCIO EXTERIOR,QUINTANA ROO,BBVA
2,NAFIN,PEQUEÑA EMPRESA,20260226,RICARDO GARCIA RAMOS,GARR-740201-910,15.17,9.642491e+05,1.928498e+06,1.0,2000000.00,2026-06-01,PRODUCTO NAFIN EMPRESARIAL,JALISCO,BBVA
3,NAFIN,PEQUEÑA EMPRESA,20250828,PANADERIA INTEGRAL SA DE CV,PIN-080716-1EA,13.81,4.264168e+05,6.091595e+05,1.0,691452.60,2026-06-01,IMPULSO REGIONAL Y REACTIVACION,SINALOA,BANORTE
4,NAFIN,MICRO EMPRESA,20250630,TIO PERFUMES DE MEXICO SA DE CV,TPM-190624-P3A,12.00,4.649504e+05,9.299008e+05,1.0,1206232.00,2026-06-01,PRODUCTO NAFIN EMPRESARIAL,TAMAULIPAS,SANTANDER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161679,NAFIN,PEQUEÑA EMPRESA,20260630,HERMILO CRUZ DURAN,CUDH-880702-PT1,14.50,6.504000e+05,8.130000e+05,1.0,813000.00,2026-06-01,TU PRIMER CREDITO,ESTADO DE MEXICO,BBVA
161680,NAFIN,MICRO EMPRESA,20260630,MAJESUS RITA BRAVO,RIBJ-900204-CD5,14.50,4.000000e+05,5.000000e+05,1.0,500000.00,2026-06-01,TU PRIMER CREDITO,MICHOACAN,BBVA
161681,NAFIN,PEQUEÑA EMPRESA,20260630,JUAN CARLOS TORRES CORTEZ,TOCJ-941203-L10,14.50,2.137600e+06,2.672000e+06,1.0,2672000.00,2026-06-01,TU PRIMER CREDITO,MICHOACAN,BBVA
161682,NAFIN,PEQUEÑA EMPRESA,20260630,COMERCIALIZADORA SERVICIOS Y SUMINISTROS INDUS...,CSS-131101-CL2,5.00,2.313561e+05,4.627123e+05,1.0,462712.27,2026-06-01,PRODUCTO NAFIN EMPRESARIAL,NUEVO LEON,BANBAJIO
